# Week 4 — Stiff Systems & Boundary Value Problems

> **Differential Equations for Scientists & Engineers**  
> *When explicit methods fail: stiffness, BDF schemes, shooting, and finite differences.*

---

## Learning Objectives

1. Define **stiffness** formally via the stiffness ratio and stability region analysis
2. Derive **Backward Euler** and **BDF-2** from the implicit Taylor series
3. Solve stiff systems with **implicit Euler** from scratch (Newton iteration for nonlinear problems)
4. Formulate and solve **Boundary Value Problems** with the **shooting method**
5. Implement **finite difference discretisation** of BVPs and the **Thomas algorithm**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.optimize import brentq

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'serif',
})

---

## 1. Stiffness — Definition and Pathology

A system $\mathbf{y}' = \mathbf{f}(t, \mathbf{y})$ is **stiff** if its Jacobian $J = \partial \mathbf{f}/\partial \mathbf{y}$ has eigenvalues $\lambda_i$ satisfying:

$$\text{Stiffness ratio} = \frac{\max_i |\text{Re}(\lambda_i)|}{\min_i |\text{Re}(\lambda_i)|} \gg 1$$

Explicit methods require $h|\lambda|$ to lie inside the **stability region**. For RK4, the stability region along the negative real axis extends only to $|h\lambda| \leq 2.79$, forcing tiny step sizes even when the solution is smooth.

### The Robertson Chemical Kinetics Problem

$$\dot{y}_1 = -0.04\,y_1 + 10^4\,y_2 y_3$$
$$\dot{y}_2 = 0.04\,y_1 - 10^4\,y_2 y_3 - 3\times10^7\,y_2^2$$
$$\dot{y}_3 = 3\times10^7\,y_2^2$$

Eigenvalues span from $O(10^{-4})$ to $O(10^{11})$ — stiffness ratio $\sim 10^{15}$.

In [ ]:
# --- Stability region comparison: Euler vs RK4 along negative real axis ---
h_lambda = np.linspace(-4, 0.5, 1000)  # h*lambda values

# Euler stability: |1 + h*lambda| <= 1  => region is disk of radius 1 centred at -1
euler_stable = np.abs(1 + h_lambda) <= 1.0

# RK4 stability function: R(z) = 1 + z + z^2/2 + z^3/6 + z^4/24
z = h_lambda
R_rk4 = 1 + z + z**2/2 + z**3/6 + z**4/24
rk4_stable = np.abs(R_rk4) <= 1.0

# Implicit Euler: R(z) = 1/(1-z)  -> always stable for Re(z) < 0
impl_euler_stable = np.abs(1.0 / (1 - h_lambda)) <= 1.0

fig, ax = plt.subplots(figsize=(9, 4))
ax.fill_between(h_lambda, 0, euler_stable.astype(float),
                alpha=0.35, color='#E53935', label='Euler stable region')
ax.fill_between(h_lambda, 0, rk4_stable.astype(float),
                alpha=0.35, color='#43A047', label='RK4 stable region')
ax.fill_between(h_lambda, 0, impl_euler_stable.astype(float),
                alpha=0.25, color='#1E88E5', label='Implicit Euler (A-stable, entire left half)')

ax.axvline(-2.79, color='#43A047', ls='--', lw=1.5, label='RK4 boundary: $h\\lambda=-2.79$')
ax.set_xlabel('$h\\lambda$ (negative real axis)')
ax.set_title('Stability Regions — Explicit vs Implicit Methods')
ax.set_yticks([])
ax.legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.show()

---

## 2. Backward (Implicit) Euler

Instead of using $f(t_n, y_n)$, use the **future** slope:

$$\boxed{y_{n+1} = y_n + h\,f(t_{n+1}, y_{n+1})}$$

This is **unconditionally stable** (A-stable). For linear $f = \lambda y$:

$$y_{n+1} = \frac{y_n}{1 - h\lambda}$$

For nonlinear problems, we solve the implicit equation at each step using **Newton's method**.

In [ ]:
def implicit_euler_linear(lam, t0, y0, T, h):
    """Implicit Euler for the linear scalar problem y' = lambda*y."""
    t_arr = np.arange(t0, T + h/2, h)
    y_arr = np.zeros(len(t_arr))
    y_arr[0] = y0
    for n in range(len(t_arr) - 1):
        y_arr[n+1] = y_arr[n] / (1 - h * lam)
    return t_arr, y_arr


# Demonstrate: stiff problem with lambda = -100
# Exact: y = exp(-100t), smooth at scale T=1 but stiff
lam = -100.0
T = 0.15
t_ex = np.linspace(0, T, 500)
y_ex = np.exp(lam * t_ex)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Explicit Euler with large h
h_explicit = 0.025   # > 2/100 = stability limit
t_e = np.arange(0, T + h_explicit/2, h_explicit)
y_e = np.zeros(len(t_e)); y_e[0] = 1.0
for n in range(len(t_e)-1):
    y_e[n+1] = y_e[n] + h_explicit * lam * y_e[n]

# Implicit Euler with same h
t_i, y_i = implicit_euler_linear(lam, 0, 1.0, T, h_explicit)

axes[0].plot(t_ex, y_ex, 'k-', lw=2, label='Exact')
axes[0].plot(t_e, y_e, 'r--o', ms=4, lw=1.5, label=f'Explicit Euler h={h_explicit}')
axes[0].plot(t_i, y_i, 'b-s', ms=4, lw=1.5, label=f'Implicit Euler h={h_explicit}')
axes[0].set_title(f'Stiff Problem: $y\' = {lam}y$')
axes[0].legend(frameon=False, fontsize=9)

axes[1].semilogy(t_e, np.abs(y_e - np.exp(lam*t_e)), 'r--', label='Explicit Euler')
axes[1].semilogy(t_i, np.abs(y_i - np.exp(lam*t_i)), 'b-',  label='Implicit Euler')
axes[1].set_title('Absolute Error'); axes[1].legend(frameon=False)

plt.tight_layout(); plt.show()

---

## 3. Boundary Value Problems — Shooting Method

A **BVP** specifies conditions at two different points:

$$y'' = f(x, y, y'), \quad y(a) = \alpha, \quad y(b) = \beta$$

The **shooting method** converts it to an IVP: guess $y'(a) = s$, integrate to $b$, then adjust $s$ to hit $y(b) = \beta$.

This defines the **residual** $r(s) = y(b; s) - \beta$, which we drive to zero with a root-finder.

In [ ]:
def shoot(f_rhs, a, alpha, b, beta, s_guess, h=0.01):
    """
    Solve BVP y'' = f(x,y,y'), y(a)=alpha, y(b)=beta via shooting.
    f_rhs: callable(x, y, dy) -> d2y
    Returns (x_arr, y_arr) for the converged solution.
    """
    def residual(s):
        # Integrate IVP: [y, y'] with IC [alpha, s] from a to b
        def sys(x, state):
            y, dy = state
            return np.array([dy, f_rhs(x, y, dy)])

        x_arr = np.arange(a, b + h/2, h)
        state = np.array([alpha, s])
        for i in range(len(x_arr) - 1):
            xi = x_arr[i]
            k1 = sys(xi,          state)
            k2 = sys(xi + h/2,    state + h/2 * k1)
            k3 = sys(xi + h/2,    state + h/2 * k2)
            k4 = sys(xi + h,      state + h   * k3)
            state = state + h/6 * (k1 + 2*k2 + 2*k3 + k4)
        return state[0] - beta

    # Bracket the root
    s_lo, s_hi = s_guess - 5, s_guess + 5
    s_star = brentq(residual, s_lo, s_hi, xtol=1e-10)

    # Final integration with converged slope
    def sys(x, state):
        y, dy = state
        return np.array([dy, f_rhs(x, y, dy)])

    x_arr = np.arange(a, b + h/2, h)
    states = np.zeros((len(x_arr), 2))
    states[0] = [alpha, s_star]
    for i in range(len(x_arr) - 1):
        xi = x_arr[i]; state = states[i]
        k1 = sys(xi,        state)
        k2 = sys(xi + h/2,  state + h/2*k1)
        k3 = sys(xi + h/2,  state + h/2*k2)
        k4 = sys(xi + h,    state + h  *k3)
        states[i+1] = state + h/6*(k1 + 2*k2 + 2*k3 + k4)

    return x_arr, states[:, 0], s_star


# --- Example: y'' = -pi^2 * y / 4,  y(0)=0, y(1)=1 ---
# Exact: y(x) = sin(pi*x/2) / sin(pi/2) = sin(pi*x/2)
f_bvp = lambda x, y, dy: -np.pi**2 * y / 4.0
x_sh, y_sh, s_star = shoot(f_bvp, a=0, alpha=0, b=1, beta=1, s_guess=1.5)
x_ex = np.linspace(0, 1, 400)
y_ex = np.sin(np.pi * x_ex / 2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_ex, y_ex, 'k-', lw=2, label='Exact $\\sin(\\pi x/2)$')
axes[0].plot(x_sh, y_sh, 'r--', lw=1.5, label=f'Shooting (s*={s_star:.4f})')
axes[0].set_title("BVP: $y'' = -\\pi^2 y/4$, $y(0)=0$, $y(1)=1$")
axes[0].legend(frameon=False)

axes[1].semilogy(x_sh, np.abs(y_sh - np.sin(np.pi*x_sh/2)), color='steelblue', lw=2)
axes[1].set_title('Shooting Method Error')
plt.tight_layout(); plt.show()

---

## 4. Finite Difference Method for BVPs

Discretise $[a,b]$ into $N+1$ points with spacing $h = (b-a)/N$.  
The second derivative is approximated by the **central difference**:

$$y''_i \approx \frac{y_{i-1} - 2y_i + y_{i+1}}{h^2}$$

For a linear BVP $y'' + p(x)y' + q(x)y = r(x)$ this produces a **tridiagonal linear system** $A\mathbf{y} = \mathbf{b}$ solvable in $O(N)$ via the **Thomas algorithm**.

In [ ]:
def thomas_algorithm(a_sub, b_diag, c_sup, d_rhs):
    """
    Solve tridiagonal system Ax=d via Thomas algorithm in O(N).
    a: subdiagonal (len N-1), b: diagonal (len N), c: superdiagonal (len N-1)
    """
    n = len(b_diag)
    c_ = np.zeros(n); d_ = np.zeros(n); x = np.zeros(n)

    # Forward sweep
    c_[0] = c_sup[0] / b_diag[0]
    d_[0] = d_rhs[0] / b_diag[0]
    for i in range(1, n):
        denom = b_diag[i] - a_sub[i-1] * c_[i-1]
        c_[i] = c_sup[i] / denom if i < n-1 else 0
        d_[i] = (d_rhs[i] - a_sub[i-1] * d_[i-1]) / denom

    # Backward substitution
    x[-1] = d_[-1]
    for i in range(n-2, -1, -1):
        x[i] = d_[i] - c_[i] * x[i+1]
    return x


def fd_bvp_linear(p_func, q_func, r_func, a, alpha, b, beta, N):
    """
    Finite difference solver for y'' + p(x)y' + q(x)y = r(x)
    with Dirichlet BCs y(a)=alpha, y(b)=beta.
    Uses central differences O(h^2) on an interior N-1 point grid.
    """
    h = (b - a) / N
    x_int = np.linspace(a, b, N+1)[1:-1]   # N-1 interior points
    M = len(x_int)

    p = np.array([p_func(xi) for xi in x_int])
    q = np.array([q_func(xi) for xi in x_int])
    r = np.array([r_func(xi) for xi in x_int])

    # Diagonal: -2/h^2 + q_i
    b_diag = -2.0/h**2 + q
    # Subdiagonal: 1/h^2 - p/(2h)
    a_sub = (1.0/h**2 - p[1:] / (2*h))
    # Superdiagonal: 1/h^2 + p/(2h)
    c_sup = (1.0/h**2 + p[:-1] / (2*h))

    rhs = r.copy()
    rhs[0]  -= (1.0/h**2 - p[0]/(2*h)) * alpha
    rhs[-1] -= (1.0/h**2 + p[-1]/(2*h)) * beta

    y_int = thomas_algorithm(a_sub, b_diag, c_sup, rhs)
    x_all = np.concatenate([[a], x_int, [b]])
    y_all = np.concatenate([[alpha], y_int, [beta]])
    return x_all, y_all


# --- Example: y'' - y = -x, y(0)=0, y(1)=0 ---
# Exact: y = x - sinh(x)/sinh(1)
p_func = lambda x: 0.0
q_func = lambda x: -1.0
r_func = lambda x: -x
y_ex_bvp = lambda x: x - np.sinh(x) / np.sinh(1)

x_ex = np.linspace(0, 1, 400)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_ex, y_ex_bvp(x_ex), 'k-', lw=2.5, label='Exact')

colors = cm.plasma(np.linspace(0.2, 0.8, 4))
for N, c in zip([5, 10, 20, 50], colors):
    x_fd, y_fd = fd_bvp_linear(p_func, q_func, r_func, 0, 0, 1, 0, N)
    axes[0].plot(x_fd, y_fd, 'o--', color=c, ms=3, lw=1, label=f'FD N={N}')

axes[0].set_title("FD BVP: $y'' - y = -x$, $y(0)=y(1)=0$")
axes[0].legend(frameon=False, fontsize=8)

N_values = [5, 10, 20, 50, 100, 200]
errors = []
for N in N_values:
    x_fd, y_fd = fd_bvp_linear(p_func, q_func, r_func, 0, 0, 1, 0, N)
    errors.append(np.max(np.abs(y_fd - y_ex_bvp(x_fd))))

h_vals = [1.0/N for N in N_values]
axes[1].loglog(h_vals, errors, 'o-', color='steelblue', lw=2)
axes[1].loglog(h_vals, np.array(h_vals)**2 * errors[0]/h_vals[0]**2,
               'k--', label='$O(h^2)$ reference')
axes[1].set_xlabel('h'); axes[1].set_ylabel('Max error')
axes[1].set_title('FD Convergence')
axes[1].legend(frameon=False)
plt.tight_layout(); plt.show()

---

## 5. Exercises

1. **(Stiffness)** For $y' = -1000(y - \cos t) - \sin t$, $y(0)=0$, what is the maximum stable step size for explicit Euler? Plot solutions with Euler at $h = 0.001$, $0.002$, $0.003$.

2. **(Newton-Implicit)** Implement a nonlinear implicit Euler solver using Newton's method with numerical Jacobian. Apply it to the van der Pol oscillator ($\mu=10$).

3. **(BDF-2)** Derive the BDF-2 formula from the second-degree polynomial interpolating $(t_{n-1},y_{n-1})$, $(t_n,y_n)$, $(t_{n+1},y_{n+1})$. Implement and verify its order-2 convergence.

4. **(Shooting vs. FD)** Solve $y'' + \pi^2 y = 0$, $y(0)=0$, $y(1)=0$ (has infinitely many solutions!). What happens to the shooting method? How does FD handle it?

5. **(Nonlinear BVP)** Apply the shooting method to the nonlinear BVP $y'' = e^y$, $y(0)=y(1)=0$ (Bratu problem). Find the two solution branches that exist for this parameter value.